# GPU Acceleration in PyTorch

This notebook demonstrates the performance benefits of GPU acceleration for neural network training in PyTorch. We'll:

1. Automatically detect available hardware (CPU, CUDA for NVIDIA GPUs, MPS for Apple Silicon)
2. Train the same model on CPU
3. Train the same model on GPU
4. Compare training times

## Hardware Detection

PyTorch supports multiple backends:
- **CPU**: Available on all systems
- **CUDA**: NVIDIA GPUs (requires CUDA toolkit)
- **MPS**: Apple Silicon GPUs (M1/M2/M3/M4 chips)

In [ ]:
import time
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
import numpy as np
import matplotlib.pyplot as plt

print(f"PyTorch version: {torch.__version__}")

In [ ]:
def get_available_devices():
    """Detect all available compute devices."""
    devices = {"cpu": torch.device("cpu")}
    
    # Check for CUDA (NVIDIA GPUs)
    if torch.cuda.is_available():
        devices["cuda"] = torch.device("cuda")
        print(f"CUDA available: {torch.cuda.get_device_name(0)}")
        print(f"  - CUDA version: {torch.version.cuda}")
        print(f"  - GPU memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
    
    # Check for MPS (Apple Silicon)
    if torch.backends.mps.is_available():
        if torch.backends.mps.is_built():
            devices["mps"] = torch.device("mps")
            print("MPS (Apple Silicon GPU) available")
        else:
            print("MPS not available: PyTorch not built with MPS support")
    
    return devices


def get_best_gpu_device(devices):
    """Return the best available GPU device, or None if only CPU available."""
    if "cuda" in devices:
        return devices["cuda"]
    elif "mps" in devices:
        return devices["mps"]
    return None


# Detect available devices
devices = get_available_devices()
gpu_device = get_best_gpu_device(devices)

print(f"\nAvailable devices: {list(devices.keys())}")
if gpu_device:
    print(f"Best GPU device: {gpu_device}")
else:
    print("No GPU available - will only run CPU benchmark")

## Create Synthetic Dataset

We'll create a synthetic classification dataset large enough to demonstrate GPU acceleration benefits. For small datasets, the overhead of transferring data to GPU can outweigh the benefits.

In [ ]:
def create_synthetic_dataset(n_samples=100000, n_features=784, n_classes=20, seed=42):
    """Create a synthetic classification dataset.
    
    Default dimensions mimic MNIST (784 features = 28x28 pixels, 10 classes).
    """
    np.random.seed(seed)
    torch.manual_seed(seed)
    
    # Generate random features
    X = np.random.randn(n_samples, n_features).astype(np.float32)
    
    # Generate labels based on a linear combination (makes it learnable)
    # Create random class centers
    centers = np.random.randn(n_classes, n_features).astype(np.float32)
    
    # Assign labels based on nearest center
    distances = np.linalg.norm(X[:, None, :] - centers[None, :, :], axis=2)
    y = distances.argmin(axis=1).astype(np.int64)
    
    # Convert to tensors
    X_tensor = torch.from_numpy(X)
    y_tensor = torch.from_numpy(y)
    
    return X_tensor, y_tensor


# Create dataset
N_SAMPLES = 300000
N_FEATURES = 784
N_CLASSES = 20
BATCH_SIZE = 256

X, y = create_synthetic_dataset(N_SAMPLES, N_FEATURES, N_CLASSES)
print(f"Dataset: {N_SAMPLES:,} samples, {N_FEATURES} features, {N_CLASSES} classes")
print(f"X shape: {X.shape}, dtype: {X.dtype}")
print(f"y shape: {y.shape}, dtype: {y.dtype}")

## Define Neural Network Model

We'll use a multi-layer perceptron (MLP) with enough parameters to benefit from GPU acceleration.

In [ ]:
class MLP(nn.Module):
    """Multi-layer perceptron for classification."""
    
    def __init__(self, input_size, hidden_sizes, num_classes, dropout=0.2):
        super().__init__()
        
        layers = []
        prev_size = input_size
        
        for hidden_size in hidden_sizes:
            layers.extend([
                nn.Linear(prev_size, hidden_size),
                nn.BatchNorm1d(hidden_size),
                nn.ReLU(),
                nn.Dropout(dropout),
            ])
            prev_size = hidden_size
        
        layers.append(nn.Linear(prev_size, num_classes))
        
        self.network = nn.Sequential(*layers)
    
    def forward(self, x):
        return self.network(x)


def count_parameters(model):
    """Count trainable parameters in a model."""
    return sum(p.numel() for p in model.parameters() if p.requires_grad)


# Model architecture
HIDDEN_SIZES = [512, 384, 256, 128]

# Create a model to check parameter count
test_model = MLP(N_FEATURES, HIDDEN_SIZES, N_CLASSES)
n_params = count_parameters(test_model)
print(f"Model architecture: {N_FEATURES} -> {' -> '.join(map(str, HIDDEN_SIZES))} -> {N_CLASSES}")
print(f"Total trainable parameters: {n_params:,}")
del test_model

## Training Function

A reusable training function that works on any device.

In [ ]:
def train_model(model, train_loader, device, n_epochs=10, lr=0.001, verbose=True):
    """Train a model on the specified device.
    
    Args:
        model: PyTorch model (will be moved to device)
        train_loader: DataLoader with training data
        device: torch.device to train on
        n_epochs: Number of training epochs
        lr: Learning rate
        verbose: Print progress
    
    Returns:
        dict with training history and timing info
    """
    # Move model to device
    model = model.to(device)
    
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam(model.parameters(), lr=lr)
    
    history = {"loss": [], "accuracy": [], "epoch_times": []}
    
    # Warm-up run (first batch can be slow due to GPU initialization)
    model.train()
    for X_batch, y_batch in train_loader:
        X_batch, y_batch = X_batch.to(device), y_batch.to(device)
        _ = model(X_batch)
        break
    
    # Synchronize before timing (important for GPU)
    if device.type == "cuda":
        torch.cuda.synchronize()
    elif device.type == "mps":
        torch.mps.synchronize()
    
    total_start = time.perf_counter()
    
    for epoch in range(n_epochs):
        epoch_start = time.perf_counter()
        
        model.train()
        running_loss = 0.0
        correct = 0
        total = 0
        
        for X_batch, y_batch in train_loader:
            # Move data to device
            X_batch = X_batch.to(device)
            y_batch = y_batch.to(device)
            
            # Forward pass
            optimizer.zero_grad()
            outputs = model(X_batch)
            loss = criterion(outputs, y_batch)
            
            # Backward pass
            loss.backward()
            optimizer.step()
            
            # Track metrics
            running_loss += loss.item() * X_batch.size(0)
            _, predicted = outputs.max(1)
            total += y_batch.size(0)
            correct += predicted.eq(y_batch).sum().item()
        
        # Synchronize before measuring time (GPU operations are async)
        if device.type == "cuda":
            torch.cuda.synchronize()
        elif device.type == "mps":
            torch.mps.synchronize()
        
        epoch_time = time.perf_counter() - epoch_start
        epoch_loss = running_loss / total
        epoch_acc = correct / total
        
        history["loss"].append(epoch_loss)
        history["accuracy"].append(epoch_acc)
        history["epoch_times"].append(epoch_time)
        
        if verbose:
            print(f"  Epoch {epoch+1}/{n_epochs}: loss={epoch_loss:.4f}, acc={epoch_acc:.4f}, time={epoch_time:.2f}s")
    
    total_time = time.perf_counter() - total_start
    history["total_time"] = total_time
    history["device"] = str(device)
    
    return history

## Train on CPU

In [ ]:
# Create DataLoader (data stays on CPU, moved to device in training loop)
dataset = TensorDataset(X, y)
train_loader = DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=True)

# Training parameters
N_EPOCHS = 10
LEARNING_RATE = 0.001

print("="*60)
print("Training on CPU")
print("="*60)

# Set random seed for reproducibility
torch.manual_seed(42)

# Create fresh model
cpu_model = MLP(N_FEATURES, HIDDEN_SIZES, N_CLASSES)

# Train on CPU
cpu_history = train_model(
    cpu_model,
    train_loader,
    device=devices["cpu"],
    n_epochs=N_EPOCHS,
    lr=LEARNING_RATE,
)

print(f"\nCPU Total training time: {cpu_history['total_time']:.2f}s")
print(f"CPU Average epoch time: {np.mean(cpu_history['epoch_times']):.2f}s")

## Train on GPU (if available)

In [ ]:
gpu_history = None

if gpu_device is not None:
    print("="*60)
    print(f"Training on GPU ({gpu_device})")
    print("="*60)
    
    # Set random seed for reproducibility
    torch.manual_seed(42)
    
    # Create fresh model
    gpu_model = MLP(N_FEATURES, HIDDEN_SIZES, N_CLASSES)
    
    # Train on GPU
    gpu_history = train_model(
        gpu_model,
        train_loader,
        device=gpu_device,
        n_epochs=N_EPOCHS,
        lr=LEARNING_RATE,
    )
    
    print(f"\nGPU Total training time: {gpu_history['total_time']:.2f}s")
    print(f"GPU Average epoch time: {np.mean(gpu_history['epoch_times']):.2f}s")
else:
    print("No GPU available - skipping GPU training")

## Compare Results

In [ ]:
print("="*60)
print("Performance Comparison")
print("="*60)

print(f"\nDataset: {N_SAMPLES:,} samples, {N_FEATURES} features")
print(f"Model: {count_parameters(MLP(N_FEATURES, HIDDEN_SIZES, N_CLASSES)):,} parameters")
print(f"Batch size: {BATCH_SIZE}, Epochs: {N_EPOCHS}")

print(f"\nCPU ({cpu_history['device']})")
print(f"  Total time: {cpu_history['total_time']:.2f}s")
print(f"  Avg epoch:  {np.mean(cpu_history['epoch_times']):.2f}s")
print(f"  Final acc:  {cpu_history['accuracy'][-1]:.4f}")

if gpu_history:
    print(f"\nGPU ({gpu_history['device']})")
    print(f"  Total time: {gpu_history['total_time']:.2f}s")
    print(f"  Avg epoch:  {np.mean(gpu_history['epoch_times']):.2f}s")
    print(f"  Final acc:  {gpu_history['accuracy'][-1]:.4f}")
    
    speedup = cpu_history['total_time'] / gpu_history['total_time']
    print(f"\nSpeedup: {speedup:.2f}x faster on GPU")

In [ ]:
# Visualize results
fig, axes = plt.subplots(1, 3, figsize=(14, 4))

epochs = range(1, N_EPOCHS + 1)

# Plot 1: Training loss
ax = axes[0]
ax.plot(epochs, cpu_history['loss'], 'b-o', label='CPU', markersize=4)
if gpu_history:
    ax.plot(epochs, gpu_history['loss'], 'r-s', label='GPU', markersize=4)
ax.set_xlabel('Epoch')
ax.set_ylabel('Loss')
ax.set_title('Training Loss')
ax.legend()
ax.grid(True, alpha=0.3)

# Plot 2: Training accuracy
ax = axes[1]
ax.plot(epochs, cpu_history['accuracy'], 'b-o', label='CPU', markersize=4)
if gpu_history:
    ax.plot(epochs, gpu_history['accuracy'], 'r-s', label='GPU', markersize=4)
ax.set_xlabel('Epoch')
ax.set_ylabel('Accuracy')
ax.set_title('Training Accuracy')
ax.legend()
ax.grid(True, alpha=0.3)

# Plot 3: Epoch times
ax = axes[2]
ax.plot(epochs, cpu_history['epoch_times'], 'b-o', label='CPU', markersize=4)
if gpu_history:
    ax.plot(epochs, gpu_history['epoch_times'], 'r-s', label='GPU', markersize=4)
ax.set_xlabel('Epoch')
ax.set_ylabel('Time (seconds)')
ax.set_title('Epoch Training Time')
ax.legend()
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# Bar chart comparison
if gpu_history:
    fig, ax = plt.subplots(figsize=(8, 5))
    
    devices_names = ['CPU', f'GPU ({gpu_device.type.upper()})']
    times = [cpu_history['total_time'], gpu_history['total_time']]
    colors = ['steelblue', 'coral']
    
    bars = ax.bar(devices_names, times, color=colors, edgecolor='black', linewidth=1.2)
    
    # Add time labels on bars
    for bar, t in zip(bars, times):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.5,
                f'{t:.2f}s', ha='center', va='bottom', fontsize=12, fontweight='bold')
    
    # Add speedup annotation
    speedup = cpu_history['total_time'] / gpu_history['total_time']
    ax.annotate(f'{speedup:.1f}x\nfaster',
                xy=(1, gpu_history['total_time']),
                xytext=(1.3, cpu_history['total_time'] * 0.6),
                fontsize=14, fontweight='bold', color='green',
                arrowprops=dict(arrowstyle='->', color='green', lw=2))
    
    ax.set_ylabel('Total Training Time (seconds)', fontsize=12)
    ax.set_title(f'CPU vs GPU Training Time\n({N_SAMPLES:,} samples, {N_EPOCHS} epochs)', fontsize=14)
    ax.set_ylim(0, max(times) * 1.2)
    
    plt.tight_layout()
    plt.show()

## Scaling Analysis: How Speedup Varies with Problem Size

GPU acceleration benefits increase with larger batch sizes and model sizes. Small problems may not benefit much due to data transfer overhead.

In [ ]:
def benchmark_batch_sizes(X, y, device, batch_sizes, n_epochs=3):
    """Benchmark training with different batch sizes."""
    results = []
    
    for batch_size in batch_sizes:
        dataset = TensorDataset(X, y)
        loader = DataLoader(dataset, batch_size=batch_size, shuffle=True)
        
        torch.manual_seed(42)
        model = MLP(N_FEATURES, HIDDEN_SIZES, N_CLASSES)
        
        history = train_model(model, loader, device, n_epochs=n_epochs, verbose=False)
        
        results.append({
            'batch_size': batch_size,
            'total_time': history['total_time'],
            'avg_epoch_time': np.mean(history['epoch_times']),
        })
        
        # Clean up
        del model
        if device.type == 'cuda':
            torch.cuda.empty_cache()
    
    return results

In [ ]:
if gpu_device is not None:
    print("Benchmarking different batch sizes...")
    print("(This may take a few minutes)\n")
    
    batch_sizes = [32, 64, 128, 256, 512, 1024]
    
    print("CPU benchmarks:")
    cpu_results = benchmark_batch_sizes(X, y, devices['cpu'], batch_sizes)
    
    print("\nGPU benchmarks:")
    gpu_results = benchmark_batch_sizes(X, y, gpu_device, batch_sizes)
    
    # Calculate speedups
    speedups = [cpu['total_time'] / gpu['total_time'] 
                for cpu, gpu in zip(cpu_results, gpu_results)]
    
    print("\nResults:")
    print(f"{'Batch Size':>12} {'CPU Time':>10} {'GPU Time':>10} {'Speedup':>10}")
    print("-" * 45)
    for cpu, gpu, speedup in zip(cpu_results, gpu_results, speedups):
        print(f"{cpu['batch_size']:>12} {cpu['total_time']:>10.2f}s {gpu['total_time']:>10.2f}s {speedup:>10.2f}x")

In [ ]:
if gpu_device is not None:
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    
    # Plot 1: Training time vs batch size
    ax = axes[0]
    ax.plot(batch_sizes, [r['total_time'] for r in cpu_results], 'b-o', label='CPU', markersize=6)
    ax.plot(batch_sizes, [r['total_time'] for r in gpu_results], 'r-s', label='GPU', markersize=6)
    ax.set_xlabel('Batch Size')
    ax.set_ylabel('Total Training Time (s)')
    ax.set_title('Training Time vs Batch Size')
    ax.set_xscale('log', base=2)
    ax.legend()
    ax.grid(True, alpha=0.3)
    
    # Plot 2: Speedup vs batch size
    ax = axes[1]
    ax.plot(batch_sizes, speedups, 'g-o', markersize=8, linewidth=2)
    ax.axhline(y=1, color='gray', linestyle='--', alpha=0.5, label='No speedup')
    ax.set_xlabel('Batch Size')
    ax.set_ylabel('Speedup (CPU time / GPU time)')
    ax.set_title('GPU Speedup vs Batch Size')
    ax.set_xscale('log', base=2)
    ax.legend()
    ax.grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.show()

## Key Takeaways

1. **Device Detection**: Use `torch.cuda.is_available()` for NVIDIA GPUs and `torch.backends.mps.is_available()` for Apple Silicon.

2. **Moving Data and Models**: Both the model and data must be on the same device:
   ```python
   model = model.to(device)
   X_batch = X_batch.to(device)
   ```

3. **Synchronization**: GPU operations are asynchronous. For accurate timing, call:
   - `torch.cuda.synchronize()` for CUDA
   - `torch.mps.synchronize()` for MPS

4. **Batch Size Matters**: Larger batch sizes typically show better GPU speedup because:
   - More parallelism to exploit
   - Lower relative overhead for data transfer
   - Better GPU utilization

5. **When GPU Helps Most**:
   - Large datasets
   - Large models (many parameters)
   - Large batch sizes
   - Matrix-heavy operations (convolutions, large linear layers)

6. **When CPU Might Be Comparable**:
   - Very small datasets or models
   - Small batch sizes
   - I/O-bound workloads
   - Operations with high branching (not easily parallelizable)